# Cuaderno 2 - Nuestro primer DataFrame con Polars

## Antes de comenzar

En el capítulo teórico vimos que Polars trabaja con estructuras tabulares que, a primera vista, nos resultan familiares. En este cuaderno vamos a llevar esas ideas a la práctica.

El objetivo no es comenzar todavía a seleccionar columnas, construir filtros o transformar datos. Eso llegará en los próximos capítulos. Aquí vamos a concentrarnos en algo más básico pero fundamental: **crear un `DataFrame`, observar cómo Polars lo representa, conocer sus dimensiones, revisar los nombres y tipos de sus columnas y cargar la misma información desde un archivo CSV**.

Como ya conocemos Pandas, varias de estas operaciones nos resultarán familiares. Conviene prestar atención, sobre todo, a la forma en que Polars muestra los tipos de datos y al concepto de **esquema**.

A lo largo del cuaderno veremos cómo:

- importar Polars;
- crear un `DataFrame` a partir de datos de Python;
- interpretar su representación;
- consultar sus dimensiones;
- revisar los nombres de sus columnas;
- consultar sus tipos y su esquema;
- observar las primeras y últimas filas;
- leer un archivo CSV;
- realizar una primera inspección de un conjunto de datos sin modificarlo.


## 1. Preparar el entorno

Google Colab puede tener Polars disponible, pero incluiremos la instalación para que el cuaderno pueda ejecutarse también en entornos donde la biblioteca todavía no esté instalada.

La opción `-q` hace que `pip` muestre menos información durante la instalación.


In [34]:
!pip install -q polars

Importamos ahora la biblioteca utilizando la abreviatura habitual `pl`.


In [35]:
import polars as pl

Podemos comprobar qué versión de Polars estamos utilizando. Este dato puede ser útil cuando un cuaderno se vuelve a ejecutar tiempo después de haber sido creado.


In [36]:
pl.__version__

'1.35.2'

## 2. Crear nuestro primer DataFrame

Trabajaremos con un conjunto pequeño de ventas de una tienda escolar. Cada fila representa un producto registrado en la tabla y las columnas describen su categoría, precio y cantidad.

El tamaño reducido es intencional: queremos poder comparar lo que muestra Polars con los datos originales sin que el volumen de información nos distraiga.


In [37]:
datos = {
    "producto": [
        "Cuaderno",
        "Lapicera",
        "Mochila",
        "Calculadora",
        "Regla",
        "Carpeta",
    ],
    "categoria": [
        "Librería",
        "Librería",
        "Accesorios",
        "Tecnología",
        "Librería",
        "Librería",
    ],
    "precio": [
        3500,
        1200,
        28500,
        18500,
        900,
        4200,
    ],
    "cantidad": [
        4,
        10,
        2,
        3,
        6,
        5,
    ],
}

df = pl.DataFrame(datos)

df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


Observamos que Polars representa el DataFrame mostrando, además de los nombres y valores, información sobre su estructura. En la parte superior aparece `shape: (6, 4)`, que indica que la tabla contiene seis filas y cuatro columnas. Debajo del nombre de cada columna también aparece su tipo de dato.

En este caso, `producto` y `categoria` fueron reconocidas como columnas de texto (`str`), mientras que `precio` y `cantidad` fueron interpretadas como números enteros (`i64`). Todavía no necesitamos modificar ninguno de esos tipos; por ahora nos interesa aprender a reconocer la estructura que Polars construyó.


## 3. Conocer las dimensiones

El atributo `shape` devuelve una tupla con la cantidad de filas y columnas, en ese orden.


In [38]:
df.shape

(6, 4)

La salida `(6, 4)` confirma que nuestro DataFrame contiene seis filas y cuatro columnas. Al consultar las dimensiones por separado, `df.height` devuelve `6` y `df.width` devuelve `4`.

Estas tres formas de consultar el tamaño describen la misma estructura desde perspectivas ligeramente diferentes: `shape` reúne ambas dimensiones, mientras que `height` y `width` permiten obtener una de ellas de manera directa.


In [39]:
df.height

6

In [40]:
df.width

4

## 4. Revisar los nombres de las columnas

El atributo `columns` nos permite conocer los nombres de las variables almacenadas en el DataFrame.


In [41]:
df.columns

['producto', 'categoria', 'precio', 'cantidad']

La salida muestra los cuatro nombres utilizados al construir el DataFrame:

```text
['producto', 'categoria', 'precio', 'cantidad']
```

Polars los devuelve como una lista de Python. En este ejemplo conocemos de antemano esas columnas porque nosotros mismos creamos los datos, pero esta comprobación será mucho más útil cuando trabajemos con archivos cuya estructura no conozcamos previamente.


## 5. Revisar los tipos de datos

Podemos obtener los tipos de todas las columnas mediante `dtypes`.


In [42]:
df.dtypes

[String, String, Int64, Int64]

La salida de `df.dtypes` es:

```text
[String, String, Int64, Int64]
```

Al relacionarla con el orden de `df.columns`, vemos que `producto` y `categoria` son columnas de texto, mientras que `precio` y `cantidad` contienen enteros de 64 bits. Esto coincide con los valores utilizados al crear el DataFrame.

A continuación podemos reunir nombres y tipos en una única estructura utilizando el **esquema**.


In [43]:
df.schema

Schema([('producto', String),
        ('categoria', String),
        ('precio', Int64),
        ('cantidad', Int64)])

El resultado de `df.schema` muestra explícitamente la correspondencia entre cada columna y su tipo:

```text
producto   → String
categoria  → String
precio     → Int64
cantidad   → Int64
```

Conviene acostumbrarnos a observar esta información desde el comienzo, porque más adelante tendrá un papel importante cuando trabajemos con conversiones de tipos, datos problemáticos y ejecución lazy. Por ahora utilizaremos el esquema como una descripción compacta de la estructura del DataFrame.


### Una pequeña prueba

Para observar cómo Polars infiere otros tipos de datos, creamos ahora un DataFrame muy pequeño con una columna de texto y otra formada por valores `True` y `False`.


In [44]:
datos_prueba = {
    "producto": ["Cuaderno", "Mochila", "Regla"],
    "disponible": [True, False, True],
}

df_prueba = pl.DataFrame(datos_prueba)

df_prueba

producto,disponible
str,bool
"""Cuaderno""",true
"""Mochila""",false
"""Regla""",true


In [45]:
df_prueba.schema

Schema([('producto', String), ('disponible', Boolean)])

La representación del nuevo DataFrame muestra `bool` debajo de `disponible`, y el esquema confirma que Polars asignó el tipo `Boolean` a esa columna. No necesitamos estudiar todavía todos los tipos disponibles; esta prueba simplemente muestra que la inferencia también reconoce de manera natural los valores booleanos.


## 6. Observar el comienzo y el final

Cuando un DataFrame contiene muchas filas no tiene sentido mostrarlo completo. `head()` permite observar sus primeros registros.


In [46]:
df.head()

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6


In [47]:
df.head(3)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2


La primera salida contiene cinco filas, que es la cantidad que `head()` muestra de manera predeterminada. Cuando escribimos `df.head(3)`, la salida se reduce a las tres primeras filas.

Estas operaciones **no modifican** `df`: producen un nuevo resultado con una cantidad limitada de registros. Podemos comprobar lo mismo desde el extremo opuesto utilizando `tail()`, que muestra las últimas filas del DataFrame.


In [48]:
df.tail()

producto,categoria,precio,cantidad
str,str,i64,i64
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


In [49]:
df.tail(2)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


### El DataFrame original permanece sin cambios

Después de utilizar `head()` y `tail()`, volvemos a consultar `df.shape`. La salida sigue siendo `(6, 4)`, de modo que el DataFrame original conserva sus seis filas y cuatro columnas. Las operaciones anteriores solo limitaron lo que se mostraba en cada resultado.


In [50]:
df.shape

(6, 4)

## 7. Guardar y volver a leer los datos

Hasta ahora construimos el DataFrame a partir de un diccionario. En el trabajo real es muy habitual recibir los datos en un archivo.

Para reproducir esa situación sin depender de una descarga externa, guardaremos momentáneamente nuestro pequeño DataFrame como `ventas.csv`. Esta escritura solo prepara el archivo que utilizaremos a continuación.


In [51]:
df.write_csv("ventas.csv")

Ahora crearemos un segundo DataFrame leyendo ese archivo con `pl.read_csv()`.


In [52]:
df_csv = pl.read_csv("ventas.csv")

df_csv

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La lectura produjo nuevamente un DataFrame de seis filas y cuatro columnas con los mismos valores que habíamos guardado. Para comprobar su estructura no necesitamos comparar manualmente cada celda: podemos utilizar las herramientas de inspección que ya conocemos.


In [53]:
df_csv.shape

(6, 4)

In [54]:
df_csv.columns

['producto', 'categoria', 'precio', 'cantidad']

In [55]:
df_csv.schema

Schema([('producto', String),
        ('categoria', String),
        ('precio', Int64),
        ('cantidad', Int64)])

In [56]:
df_csv.head(3)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2


## 8. Construir una primera lectura del dataset

Podemos pensar ahora en `ventas.csv` como si fuera un archivo recibido desde otra persona o sistema. Antes de comenzar cualquier análisis, una inspección inicial con `shape`, `columns`, `schema`, `head()` y `tail()` permite obtener una descripción bastante completa de su estructura sin modificar los datos.


In [57]:
df_csv.shape

(6, 4)

In [58]:
df_csv.columns

['producto', 'categoria', 'precio', 'cantidad']

In [59]:
df_csv.schema

Schema([('producto', String),
        ('categoria', String),
        ('precio', Int64),
        ('cantidad', Int64)])

In [60]:
df_csv.head()

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6


In [61]:
df_csv.tail()

producto,categoria,precio,cantidad
str,str,i64,i64
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


### Lo que podemos concluir de esta primera inspección

Las salidas anteriores muestran que `ventas.csv` contiene **6 filas y 4 columnas**. Las variables son `producto`, `categoria`, `precio` y `cantidad`.

El esquema indica que `producto` y `categoria` son columnas de texto (`String`), mientras que `precio` y `cantidad` son columnas numéricas enteras (`Int64`). Las primeras y últimas filas también muestran valores coherentes con el contenido que habíamos guardado, por lo que la lectura del CSV conservó correctamente la estructura de los datos.

Todavía no estamos obteniendo conclusiones sobre las ventas. Esta primera revisión se limita a describir la **estructura** del conjunto de datos y a comprobar que la carga produjo un resultado razonable.


## 9. Qué conviene recordar

En este cuaderno no analizamos todavía los valores de las ventas. Nuestro trabajo consistió en aprender a reconocer la estructura de un DataFrame de Polars.

Las herramientas utilizadas fueron:

```text
df.shape
df.height
df.width
df.columns
df.dtypes
df.schema
df.head()
df.tail()
pl.read_csv(...)
```

No es necesario memorizar la lista como si fueran instrucciones aisladas. Conviene asociar cada herramienta con una pregunta:

| Pregunta | Herramienta |
|---|---|
| ¿Cuántas filas y columnas hay? | `shape` |
| ¿Cuántas filas hay? | `height` |
| ¿Cuántas columnas hay? | `width` |
| ¿Cómo se llaman las columnas? | `columns` |
| ¿Qué tipos tienen? | `dtypes` |
| ¿Qué nombre y tipo tiene cada columna? | `schema` |
| ¿Cómo son los primeros registros? | `head()` |
| ¿Cómo son los últimos registros? | `tail()` |
| ¿Cómo leo un CSV? | `pl.read_csv()` |

Esta inspección no reemplaza un análisis posterior, pero nos da una primera descripción del conjunto de datos y nos permite comprobar que su estructura es razonable antes de comenzar a trabajar con él.


## Próximo paso

Hasta aquí nos mantuvimos deliberadamente en un terreno bastante familiar. Observamos el DataFrame, sus dimensiones, sus columnas y sus tipos, pero todavía no intentamos seleccionar partes específicas ni transformar información.

En el próximo capítulo veremos una diferencia importante con el modelo que conocemos de Pandas: Polars no organiza el trabajo alrededor de un índice de filas equivalente al de aquella biblioteca. Antes de comenzar con las selecciones mediante expresiones, convendrá entender qué cambia y por qué no debemos buscar simplemente una copia de `loc` o `iloc`.
